Splitting labels into two groups fake news and reliable.

In [36]:
import pandas as pd
import ast
from rich.progress import track
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import linear_model
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
import swifter
import numpy as np

In [15]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

training_set = pd.read_csv('../Group_project/training_set.csv', converters = converters_dict)

In [19]:
# creating list of 10000 most common words from vocabulary found in part 1, the first row has been removed due to it being 'na'
full_voc = pd.read_csv('../Group_project/vocabulary_995000.csv')
sub_voc = set((full_voc.iloc[:10000])['word'].values.tolist())
print(sub_voc)

{'planet', 'sexiest', 'gum', 'consecut', 'oc', 'panel', 'audit', 'pen', 'conservapedia', 'subscrib', 'consult', 'plot', 'terribl', 'buzz', 'interst', 'cuisin', 'guam', 'florenc', 'remot', 'synchron', 'rs', 'musician', 'vernon', 'intrins', 'contain', 'assort', 'carv', 'radic', 'news', 'patterson', 'homophob', 'valley', 'statut', 'chosen', 'turkish', 'peanut', 'irrelev', 'censorship', 'issuesuprem', 'palestin', 'grammi', 'malcolm', 'chest', 'vitamin', 'pedophil', 'peninsula', 'influenti', 'milan', 'dig', 'tomb', 'seed', 'dominican', 'markl', 'chomski', 'altar', 'dmitri', 'wedg', 'burglari', 'gov', 'massag', 'fusion', 'utah', 'malwar', 'doom', 'delusion', 'embryo', 'glorious', 'repudi', 'institut', 'employ', 'undercov', 'mask', 'usag', 'farc', 'digit', 'homer', 'glass', 'buyer', 'mead', 'undergo', 'colorado', 'newborn', 'restaur', 'sweet', 'er', 'india', 'fiber', 'poke', 'paint', 'barrel', 'request', 'run', 'nullifi', 'glitch', 'griev', 'anita', 'elian', 'habit', 'relat', 'scene', 'zero',

In [20]:
types = {i[0] for i in training_set['type'] if len(i) < 2}
print(types)
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'clickbait', 'unreli', 'polit', 'reliabl'}

def filter_type(type, fake, rel):
    if type in fake:
        return 'fake'
    elif type in rel:
        return 'reliable'

{'fake', 'junksci', 'unknown', 'clickbait', 'hate', 'bias', 'rumor', 'unreli', 'conspiraci', 'satir', 'reliabl', 'polit'}


In [41]:
def filter_strings(str_list, **kwargs):
    filtered = [word for word in str_list if word in sub_voc]
    joined = ' '.join(filtered)
    return joined

relevant_columns = [
    'content', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']

def filter_df(df):
    f_df = pd.DataFrame({})
    for i in track(relevant_columns, 'processing...'):
        f_df[i] = df[i].swifter.progress_bar(False).apply(filter_strings)
    return f_df

filtered = filter_df(training_set)

filtered['y_t'] = training_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))
print(filtered)

Output()

                                                  content  \
0       articl googl ali alfoneh assist compil politic...   
1       texa stand religi belief alleg teacher forc st...   
2       black kos editor dr woman african american ame...   
3       amaz moment gorilla reunit human friend back r...   
4       miss time alien disc bitcoin blockchain search...   
...                                                   ...   
795995  roger tlb don talk hear news exist danger chil...   
795996  le du du moham abdel sur son twitter action du...   
795997  arkansa gov mike huckabe call christian pastor...   
795998  news north korea latest attempt world attent a...   
795999  pm est updat comput firm post loss compar resu...   

                                                    title  \
0                                         iran news round   
1       video student tx teacher forc grader deni god ...   
2                                   black kos week review   
3                      

In [38]:
x_t = filtered[filtered['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
y_t = x_t['y_t']

transformer_list = [(f'vec_{col}', CountVectorizer(), col) for col in relevant_columns]
transformer = ColumnTransformer(transformers=transformer_list, remainder='drop')

X_train = transformer.fit_transform(x_t)
y_train = np.array(y_t)

In [39]:
logr = linear_model.LogisticRegression(max_iter = 100000)

logr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [40]:
test_set = pd.read_csv('../Group_project/test_set.csv', converters = converters_dict)

In [ ]:
filtered_test = filter_df(test_set)

filtered_test['y_t'] = test_set['type'].swifter.progress_bar(False).apply(lambda x, **kwargs: filter_type(x[0], fake_news, reliable_news))

Output()

In [47]:
x_test = filtered_test[filtered_test['y_t'].apply(lambda x, **kwargs: x in {'fake', 'reliable'})]
yt_t = x_test['y_t']

X_test = transformer.transform(x_test)
y_test = np.array(yt_t)

In [48]:
predictions = logr.predict(X_test)

In [49]:
print(accuracy_score(y_test, predictions))

0.9594238227146814
